## Fraud Analytics - Assignment 1

### Hitarth Parikh - 25030242024

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler,StandardScaler,LabelEncoder,OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from collections import Counter
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler, SMOTE, BorderlineSMOTE, ADASYN



In [35]:
df = pd.read_csv("Fraud Detection Dataset.csv")

In [36]:
print(list(df.columns))

['Transaction_ID', 'User_ID', 'Transaction_Amount', 'Transaction_Type', 'Time_of_Transaction', 'Device_Used', 'Location', 'Previous_Fraudulent_Transactions', 'Account_Age', 'Number_of_Transactions_Last_24H', 'Payment_Method', 'Fraudulent']


### Partition

In [39]:
split_fraction = 0.10
partition = df.sample(frac=split_fraction, random_state=42)
len(partition)

5100

### Random 

In [40]:
sample_random = df.sample(n=500)
sample_random.head()

,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Time_of_Transaction,Device_Used,Location,Previous_Fraudulent_Transactions,Account_Age,Number_of_Transactions_Last_24H,Payment_Method,Fraudulent
6574,T6575,4293,3353.20,POS Payment,0.0,Mobile,NaN,1,2,10,Invalid Method,0
26425,T26426,3610,NaN,POS Payment,11.0,Desktop,NaN,4,10,7,UPI,0
31579,T31580,4433,3326.13,Online Purchase,19.0,Tablet,Seattle,0,119,9,Net Banking,0
8813,T8814,2554,3493.95,Bill Payment,23.0,Desktop,Los Angeles,4,57,10,Net Banking,0
43885,T43886,4952,4917.86,ATM Withdrawal,7.0,NaN,Boston,4,10,12,UPI,0


### Task 1: Fix Number of Sample (500 only)

In [44]:
# Multiple ways to do it 
# 1. Random Sampling
fraud_data = df.sample(n=500, random_state=42)
fraud_data.head()

,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Time_of_Transaction,Device_Used,Location,Previous_Fraudulent_Transactions,Account_Age,Number_of_Transactions_Last_24H,Payment_Method,Fraudulent
4207,T4208,2440,1914.57,Online Purchase,21.0,Mobile,Seattle,1,40,12,Debit Card,0
6583,T6584,4693,2987.32,POS Payment,8.0,Desktop,Boston,2,85,13,Credit Card,0
13538,T13539,3457,1338.51,Bill Payment,14.0,Desktop,Boston,3,92,10,Credit Card,0
38447,T38448,1111,3142.30,ATM Withdrawal,NaN,Desktop,Seattle,4,47,9,Credit Card,0
22267,T22268,3178,NaN,POS Payment,16.0,Mobile,Los Angeles,3,61,14,Debit Card,0


In [45]:
# 2. Select every 20th record to get 500 samples
fraud_data = df.iloc[::20].head(500)

print(len(fraud_data))

500


### Task 2: Location -wise list/count

In [46]:
# Create a location-wise count table

location_counts = fraud_data['Location'].value_counts().reset_index()

location_counts.columns = ['Location', 'Count']

print(location_counts)

        Location  Count
0          Miami     69
1        Chicago     69
2  San Francisco     63
3        Houston     61
4       New York     60
5    Los Angeles     59
6         Boston     58
7        Seattle     38


### Task 3: Equalize the location wise count

In [47]:
# Equalize the number of transactions for each location

min_count = fraud_data['Location'].value_counts().min()

equalized_data = (
    fraud_data.groupby('Location', group_keys=False)
    .apply(lambda x: x.sample(n=min_count, random_state=42))
)

print(equalized_data['Location'].value_counts())

Location
Boston           38
Chicago          38
Houston          38
Los Angeles      38
Miami            38
New York         38
San Francisco    38
Seattle          38
Name: count, dtype: int64


C:\Users\Hitarth Parikh\AppData\Local\Temp\ipykernel_31240\799471286.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min_count, random_state=42))


### Task 4: Sample from 5 most/least listed countries  

In [48]:
# Find the 5 most and 5 least listed locations

location_counts = fraud_data['Location'].value_counts()

most_5 = location_counts.head(5)
least_5 = location_counts.tail(5)

print("5 Most Listed Locations:")
print(most_5)

print("\n5 Least Listed Locations:")
print(least_5)

5 Most Listed Locations:
Location
Miami            69
Chicago          69
San Francisco    63
Houston          61
New York         60
Name: count, dtype: int64

5 Least Listed Locations:
Location
Houston        61
New York       60
Los Angeles    59
Boston         58
Seattle        38
Name: count, dtype: int64


In [50]:
# Sample 10 transactions from the 5 most and 5 least listed locations

selected_locations = list(most_5.index) + list(least_5.index)

sampled_data = (
    fraud_data[fraud_data['Location'].isin(selected_locations)]
    .groupby('Location', group_keys=False)
    .apply(lambda x: x.sample(n=min(10, len(x)), random_state=42))
)

C:\Users\Hitarth Parikh\AppData\Local\Temp\ipykernel_31240\2254231659.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(10, len(x)), random_state=42))


In [52]:
sampled_data.head()

,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Time_of_Transaction,Device_Used,Location,Previous_Fraudulent_Transactions,Account_Age,Number_of_Transactions_Last_24H,Payment_Method,Fraudulent
100,T101,3449,1755.29,Online Purchase,15.0,Tablet,Boston,1,9,1,Credit Card,0
620,T621,3849,4090.28,Bill Payment,16.0,Desktop,Boston,0,34,3,Credit Card,0
6560,T6561,1370,288.93,Bank Transfer,4.0,Mobile,Boston,1,76,8,Net Banking,0
2600,T2601,1548,1602.41,ATM Withdrawal,13.0,Mobile,Boston,3,92,1,UPI,0
7540,T7541,2939,937.44,ATM Withdrawal,NaN,Desktop,Boston,1,76,5,Invalid Method,1


### Task 5: ML Model

In [53]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Create a copy of the dataset
ml_data = fraud_data.copy()

# Remove ID columns and separate target variable
X = ml_data.drop(['Fraudulent', 'Transaction_ID', 'User_ID'], axis=1)
y = ml_data['Fraudulent']

# Convert categorical columns into numerical values
categorical_cols = X.select_dtypes(include='object').columns

for col in categorical_cols:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Create and train the Random Forest model
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.95

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        95
           1       0.00      0.00      0.00         5

    accuracy                           0.95       100
   macro avg       0.47      0.50      0.49       100
weighted avg       0.90      0.95      0.93       100



C:\Users\Hitarth Parikh\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Hitarth Parikh\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Hitarth Parikh\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, 